# 01_Narrow_Transformations — select, filter, withColumn, drop, union; RDD map/flatMap

**Created:** 2026-02-05 04:59:58

> This notebook teaches PySpark step‑by‑step using the tiny array `123456781`. Everything is explained like you're 5: small stories, simple words, and prints everywhere so you can *see* what happens.


## 0) Spark Setup (ELI5)

- Think of Spark like a **kitchen**. The **driver** is the chef that reads the recipe. The **executors** are helpers who actually chop, stir, and cook.
- We ask the chef to **plan** the recipe (lazy *transformations*), and we only **cook** when we shout "Serve!" (an *action*).

Run the next cell to make a tiny kitchen on your laptop (`local[*]`). If PySpark isn't installed, the cell will tell you how to install it.


In [ ]:

# SparkSession is the doorway to Spark SQL/DataFrames
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql import types as T
    from pyspark.sql.window import Window
    spark = (SparkSession
             .builder
             .appName("ELI5-PySpark-Operators")
             .master("local[*]")  # use all local cores
             .getOrCreate())
    print("✅ Spark is ready:", spark.version)
except Exception as e:
    print("❌ PySpark not available in this environment.")
    print("Install with: pip install pyspark --quiet")
    raise



## 1) Our tiny toys (data)

We turn `123456781` into:
- An **RDD** of digits (low-level Spark collection), and
- A **DataFrame** with one column `x` (higher-level table‑like structure).

We also add a tiny **index** column to make sorting/join demos easier.


In [ ]:

# Our raw Python list and string
py_list = [1,2,3,4,5,6,7,8,1]
py_str  = "123456781"

# Create an RDD from the list
rdd = spark.sparkContext.parallelize(py_list, 2)  # 2 partitions so you can see partition behavior
print("RDD partitions:", rdd.getNumPartitions())
print("RDD sample:", rdd.take(5))

# Create a DataFrame with an index
df = spark.createDataFrame([(i, v) for i, v in enumerate(py_list)], schema=["idx","x"]) 
print("DataFrame schema:")
df.printSchema()
print("First rows:")
df.show(5, truncate=False)



## 2) Narrow Transformations (no shuffle)
**ELI5:** Each helper (executor) can finish its plate without swapping food with others.

We'll cover:
- `select()` — pick columns
- `filter()/where()` — keep some rows
- `withColumn()` — make or change a column
- `withColumnRenamed()` — rename a column
- `drop()` — remove a column
- `union()` — stack two DataFrames with same columns
- RDD `map()` / `flatMap()` — apply a function to each item


In [ ]:

from pyspark.sql import functions as F

print("Start df:")
df.show()

# select: choose columns (like choosing only the red blocks)
only_x = df.select("x")
print("only_x = df.select('x')
This creates a new DataFrame with just the 'x' column.")
only_x.show()

# filter / where: keep rows that match a condition
bigger_than_4 = df.where(F.col("x") > 4)  # alias: df.filter(...)
print("Rows where x > 4:")
bigger_than_4.show()

# withColumn: create/replace a column
labeled = df.withColumn("label", F.when(F.col("x") % 2 == 0, F.lit("even")).otherwise(F.lit("odd")))
print("Add 'label' column that says even/odd:")
labeled.show()

# withColumnRenamed: rename
renamed = labeled.withColumnRenamed("label", "parity")
print("Renamed 'label' -> 'parity':")
renamed.show()

# drop: remove a column
no_idx = renamed.drop("idx")
print("Dropped 'idx':")
no_idx.show()

# union: stack two DataFrames (schemas must match)
print("Union the DataFrame with itself (doubles the rows):")
stacked = df.select("idx","x").union(df.select("idx","x"))
stacked.show(5)
print("Row count after union:", stacked.count())

# RDD map: apply a function to each element
print("
RDD map: add 10 to each number")
print(rdd.map(lambda n: n + 10).take(5))

# RDD flatMap: split into many
print("RDD flatMap: for each number n, create list [n, n*n]")
print(rdd.flatMap(lambda n: [n, n*n]).take(10))



### Mini‑labs
1) Keep only rows where `x` is in `{2,4,6,8}` using `isin`.
2) Create a new column `x2 = x * x` and then drop it.
3) Build `rdd2 = rdd.map(lambda n: (n % 2, 1))` and explain what each pair means.
